# Open-Vocabulary Image Annotation with Grounding DINO and SAM 2

_Authored by: [sanobar20](https://github.com/sanobar20)_

Most object detectors can only find the classes they were trained on. In this recipe we build a
**training-free annotation pipeline**: you describe the objects you care about in plain English,
[Grounding DINO](https://huggingface.co/IDEA-Research/grounding-dino-tiny) draws the boxes, and
[SAM 2](https://huggingface.co/facebook/sam2.1-hiera-tiny) turns each box into a pixel-accurate mask.

The result is a COCO-style annotation file that you can review, correct, and use to bootstrap a
custom dataset — no labels and no fine-tuning required.

## What you'll build

```
image + text prompts -> Grounding DINO -> boxes -> SAM 2 -> masks -> COCO JSON
```

By the end of this notebook you will be able to:

- detect arbitrary object classes from a text prompt;
- understand the two thresholds that control detection quality;
- convert boxes into masks with SAM 2;
- batch-process several images and export the annotations.

**Requirements:** a GPU is recommended but not required. We use the "tiny" checkpoints so the
notebook fits on a free Colab T4 and still runs on CPU for a couple of images.

## Setup

We need a recent `transformers` (for SAM 2), plus `pycocotools` for the COCO export.

In [ ]:
!pip install -q -U transformers pycocotools


## Imports and device

In [ ]:
import json
import requests
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle
from PIL import Image

from transformers import (
    AutoModelForZeroShotObjectDetection,
    AutoProcessor,
    Sam2Model,
    Sam2Processor,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Load the models

- **Grounding DINO Tiny** is an open-vocabulary detector: it scores boxes against a text prompt.
- **SAM 2.1 Hiera Tiny** is a promptable segmenter: give it a box (or a point), get a mask.

Both checkpoints are small enough for free-tier hardware. We put them in `eval()` mode because we
only run inference.

The SAM 2.1 checkpoints on the Hub are published as `sam2_video` models. `Sam2Model` loads their
image encoder and mask decoder, which is exactly what we need, so `transformers` prints a one-line
notice about the checkpoint type while loading. That notice is expected.


In [ ]:
DETECTOR_ID = "IDEA-Research/grounding-dino-tiny"
SEGMENTER_ID = "facebook/sam2.1-hiera-tiny"

detector_processor = AutoProcessor.from_pretrained(DETECTOR_ID)
detector = AutoModelForZeroShotObjectDetection.from_pretrained(DETECTOR_ID).to(device).eval()

sam_processor = Sam2Processor.from_pretrained(SEGMENTER_ID)
sam = Sam2Model.from_pretrained(SEGMENTER_ID).to(device).eval()

print("Models loaded.")

## Sample images and prompts

We use two COCO validation images and the SAM 2 "truck" fixture. Each sample pairs an image with
the classes we want to find. Try replacing these with your own images and prompts.

In [ ]:
SAMPLES = [
    {
        "name": "cats",
        "url": "http://images.cocodataset.org/val2017/000000039769.jpg",
        "classes": ["a cat", "a remote control"],
    },
    {
        "name": "living_room",
        "url": "http://images.cocodataset.org/val2017/000000000139.jpg",
        "classes": ["a dining table", "a chair", "a vase"],
    },
    {
        "name": "truck",
        "url": "https://huggingface.co/datasets/hf-internal-testing/sam2-fixtures/resolve/main/truck.jpg",
        "classes": ["a truck", "a wheel"],
    },
]

images = []
for sample in SAMPLES:
    image = Image.open(requests.get(sample["url"], stream=True).raw).convert("RGB")
    images.append(image)
    print(f"{sample['name']}: {image.size[0]}x{image.size[1]}")

In [ ]:
fig, axes = plt.subplots(1, len(images), figsize=(15, 4))
for ax, image, sample in zip(axes, images, SAMPLES):
    ax.imshow(image)
    ax.set_title(sample["name"])
    ax.axis("off")
plt.tight_layout()

## Step 1 — text-prompted detection

Grounding DINO takes the class list as one period-separated string, for example
`"a cat. a remote control."`. Each phrase between the dots is scored independently, so short,
concrete noun phrases give the cleanest boxes.

Two thresholds matter:

- `box_threshold` filters boxes by their overall objectness score.
- `text_threshold` filters boxes by how well they match the text prompt.

Lower values give more boxes (higher recall, more false positives). Higher values give fewer,
more confident boxes.


In [ ]:
def detect_objects(image, classes, box_threshold=0.35, text_threshold=0.25):
    """Return boxes, scores, and labels for a list of text prompts."""
    text = ". ".join(c.strip().rstrip(".") for c in classes) + "."
    inputs = detector_processor(images=image, text=text, return_tensors="pt").to(device)

    with torch.inference_mode():
        outputs = detector(**inputs)

    result = detector_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=[image.size[::-1]],
    )[0]

    # `text_labels` can contain a stray empty string when nothing was detected,
    # so trim it to the number of boxes.
    labels = [label.strip() for label in result["text_labels"]][: len(result["scores"])]
    # Drop boxes whose phrase did not survive `text_threshold`, since they would
    # otherwise become an annotation with an empty category name.
    keep = torch.tensor([bool(label) for label in labels], dtype=torch.bool)
    labels = [label for label in labels if label]

    # Sort by confidence so the best boxes appear first.
    order = torch.argsort(result["scores"][keep], descending=True)
    return {
        "boxes": result["boxes"][keep][order],
        "scores": result["scores"][keep][order],
        "labels": [labels[int(i)] for i in order],
    }


result = detect_objects(images[0], SAMPLES[0]["classes"])
for box, score, label in zip(result["boxes"], result["scores"], result["labels"]):
    coords = [round(float(v), 1) for v in box]
    print(f"{label:20s} {float(score):.3f}  {coords}")


### Visualize the boxes

In [ ]:
def draw_boxes(image, boxes, scores, labels, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(image)
    ax.axis("off")
    colors = plt.cm.tab20(np.linspace(0, 1, max(len(boxes), 1)))
    for box, score, label, color in zip(boxes, scores, labels, colors):
        x1, y1, x2, y2 = [float(v) for v in box]
        ax.add_patch(
            Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2)
        )
        ax.text(
            x1,
            max(y1 - 5, 0),
            f"{label} {float(score):.2f}",
            fontsize=9,
            color="white",
            bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none"),
        )
    return ax

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, image, sample in zip(axes, images, SAMPLES):
    result = detect_objects(image, sample["classes"])
    draw_boxes(image, result["boxes"], result["scores"], result["labels"], ax=ax)
    ax.set_title(f"{sample['name']} ({len(result['boxes'])} boxes)")
plt.tight_layout()

## Step 2 — how the thresholds change the result

The same prompt can produce a handful of boxes or dozens depending on the thresholds. Sweeping
`box_threshold` on one image is a quick way to see the precision/recall trade-off before you
commit to a value for a batch.

In [ ]:
for threshold in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    result = detect_objects(images[0], SAMPLES[0]["classes"], box_threshold=threshold)
    counts = {label: result["labels"].count(label) for label in set(result["labels"])}
    print(f"box_threshold={threshold:.2f}  total={len(result['boxes'])}  per_class={counts}")

In practice, `0.30–0.40` is a good starting range for the tiny checkpoint. Use a lower
`text_threshold` when your prompt is a long phrase and a higher one when you want strict
matching. Always inspect a few images before using a threshold on a whole dataset.


## Step 3 — from boxes to masks with SAM 2

SAM 2 takes the boxes as prompts. For each box it returns a mask for that object. Passing
`multimask_output=False` gives one mask per box, which is what we want for annotation.

In [ ]:
def segment_boxes(image, boxes):
    """Return one boolean mask per box, at the image's original resolution."""
    if len(boxes) == 0:
        return torch.zeros((0, image.height, image.width), dtype=torch.bool)

    input_boxes = [[box.tolist() for box in boxes]]
    inputs = sam_processor(images=image, input_boxes=input_boxes, return_tensors="pt").to(device)

    with torch.inference_mode():
        outputs = sam(**inputs, multimask_output=False)

    masks = sam_processor.post_process_masks(outputs.pred_masks.cpu(), inputs["original_sizes"])[0]
    if masks.ndim == 3:
        masks = masks.unsqueeze(1)
    return masks[:, 0].bool()


def show_masks(image, boxes, scores, labels, masks, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(image)
    ax.axis("off")
    colors = plt.cm.tab20(np.linspace(0, 1, max(len(boxes), 1)))
    for box, score, label, mask, color in zip(boxes, scores, labels, masks, colors):
        mask_np = mask.cpu().numpy() if torch.is_tensor(mask) else np.asarray(mask)
        overlay = np.zeros((*mask_np.shape, 4), dtype=float)
        overlay[mask_np] = [color[0], color[1], color[2], 0.45]
        ax.imshow(overlay)
        x1, y1, x2, y2 = [float(v) for v in box]
        ax.add_patch(
            Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2)
        )
        ax.text(
            x1,
            max(y1 - 5, 0),
            f"{label} {float(score):.2f}",
            fontsize=9,
            color="white",
            bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none"),
        )
    return ax

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, image, sample in zip(axes, images, SAMPLES):
    result = detect_objects(image, sample["classes"])
    masks = segment_boxes(image, result["boxes"])
    show_masks(image, result["boxes"], result["scores"], result["labels"], masks, ax=ax)
    ax.set_title(f"{sample['name']} ({len(result['boxes'])} masks)")
plt.tight_layout()

## Step 4 — a reusable annotation function

Now we can wrap detection and segmentation into one function and run it over the batch. The
function returns tensors and plain strings, so it is easy to reuse outside the notebook.

In [ ]:
def annotate(image, classes, box_threshold=0.35, text_threshold=0.25):
    """Detect `classes` in `image` and return boxes, scores, labels, and masks."""
    result = detect_objects(image, classes, box_threshold, text_threshold)
    masks = segment_boxes(image, result["boxes"])
    return {
        "boxes": result["boxes"].cpu(),
        "scores": result["scores"].cpu(),
        "labels": result["labels"],
        "masks": masks,
    }


annotations = []
for image, sample in zip(images, SAMPLES):
    ann = annotate(image, sample["classes"])
    annotations.append(ann)
    print(
        f"{sample['name']}: {len(ann['boxes'])} objects, "
        f"{int(ann['masks'].sum())} mask pixels"
    )

## Step 5 — export COCO annotations

COCO is a good interchange format: most annotation tools and evaluation libraries can read it.
Boxes are stored as `[x, y, width, height]`, and masks are stored as run-length encoded (RLE)
segmentation masks.

In [ ]:
from pycocotools import mask as mask_utils


def mask_to_rle(mask):
    mask = mask.cpu().numpy() if torch.is_tensor(mask) else np.asarray(mask)
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle


coco = {
    "info": {"description": "Open-vocabulary annotations from Grounding DINO + SAM 2"},
    "images": [],
    "annotations": [],
    "categories": [],
}
category_ids = {}
next_ann_id = 1

for image_id, (image, sample, ann) in enumerate(zip(images, SAMPLES, annotations)):
    coco["images"].append(
        {
            "id": image_id,
            "file_name": f"{sample['name']}.jpg",
            "width": image.width,
            "height": image.height,
        }
    )

    for box, score, label, mask in zip(
        ann["boxes"], ann["scores"], ann["labels"], ann["masks"]
    ):
        if label not in category_ids:
            category_ids[label] = len(category_ids) + 1
            coco["categories"].append(
                {"id": category_ids[label], "name": label, "supercategory": "object"}
            )

        x1, y1, x2, y2 = [float(v) for v in box]
        coco["annotations"].append(
            {
                "id": next_ann_id,
                "image_id": image_id,
                "category_id": category_ids[label],
                "bbox": [x1, y1, x2 - x1, y2 - y1],
                "area": float(mask.sum()),
                "segmentation": mask_to_rle(mask),
                "iscrowd": 0,
                "score": float(score),
            }
        )
        next_ann_id += 1

Path("annotations.json").write_text(json.dumps(coco, indent=2))
print(f"Wrote {len(coco['images'])} images and {len(coco['annotations'])} annotations.")

In [ ]:
from pycocotools.coco import COCO

check = COCO("annotations.json")
print(f"COCO loaded: {len(check.imgs)} images, {len(check.anns)} annotations")

## Limitations and next steps

This is a bootstrapping tool, not a replacement for reviewed labels. Keep these in mind:

- **Prompt sensitivity.** Concrete noun phrases work best ("a person wearing a red jacket"
  rather than "red jacket"). Test several phrasings on a few images before scaling up.
- **Duplicate boxes.** Grounding DINO can return overlapping boxes for the same object. Apply
  per-class non-maximum suppression, or deduplicate by mask IoU, before exporting.
- **Loose boxes make loose masks.** SAM 2 segments what is inside the box, so an undersized or
  oversized box directly hurts mask quality.
- **Small and occluded objects** are the hardest cases; expect lower recall there.
- **Zero-shot accuracy is not production accuracy.** Review and correct the output, then use it
  to train or fine-tune a supervised model.

Practical extensions:

1. Add per-class NMS with `torchvision.ops.nms`.
2. Push the reviewed dataset to the Hub and inspect it in FiftyOne or Label Studio.
3. Use the annotations for active learning: train on the confident subset, then re-run the
   pipeline on the unlabeled pool.
4. Fine-tune Grounding DINO or a closed-set detector once you have enough reviewed labels.